<a href="https://colab.research.google.com/github/IsaC1B/hotel-booking-chatbot---Lab-Software/blob/Google/simulador_chatbot_hotel_gradio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 Simulador de Chatbot IA — Hotel

Notebook para **Google Colab** que simula un chatbot web para un sistema de reservas de **alojamiento y alimentos**.

### Historias de usuario cubiertas
- HU01: consultar horarios del hotel.
- HU02: consultar opciones de alimentación.
- HU03: salir del chatbot.
- HU04: consultar disponibilidad de habitaciones por fechas.
- HU05: consultar servicios y características de habitaciones.

### Requisitos funcionales cubiertos
RF01–RF05, además de normalización de entradas, separación de configuración y credenciales, documentación y una interfaz web mediante Gradio.

> **Nota:** este notebook implementa un simulador inteligente basado en reglas e identificación de intenciones. Es apropiado para un prototipo académico; posteriormente puede conectarse a un modelo LLM y a una base de datos real de reservas.

## 1. Instalación de dependencias

Ejecuta esta celda una vez en Google Colab.

In [ ]:
!pip -q install gradio>=5.0.0


## 2. Importaciones y configuración

La información del hotel utilizada en el prototipo es ficticia. En una implementación real, estos datos deberían provenir de una fuente segura y actualizable.

In [ ]:
import re
import unicodedata
from datetime import datetime
import gradio as gr

# ============================================================
# CONFIGURACIÓN DEL PROTOTIPO
# No colocar contraseñas, API keys ni credenciales aquí.
# ============================================================

HOTEL = {
    "nombre": "Hotel Montaña Verde",
    "check_in": "15:00",
    "check_out": "12:00",
    "desayuno": "06:30 - 09:30",
    "restaurante": "12:00 - 21:00",
}

HABITACIONES = {
    "individual": {
        "nombre": "Habitación Individual",
        "capacidad": 1,
        "precio": 150000,
        "servicios": ["Wi-Fi", "TV", "baño privado", "escritorio"]
    },
    "doble": {
        "nombre": "Habitación Doble",
        "capacidad": 2,
        "precio": 220000,
        "servicios": ["Wi-Fi", "TV", "baño privado", "aire acondicionado"]
    },
    "familiar": {
        "nombre": "Habitación Familiar",
        "capacidad": 4,
        "precio": 320000,
        "servicios": ["Wi-Fi", "TV", "baño privado", "aire acondicionado", "minibar"]
    }
}

# Disponibilidad ficticia para el simulador.
# En producción debe consultarse una base de datos/API de reservas.
DISPONIBILIDAD = {
    "individual": True,
    "doble": True,
    "familiar": False,
}


## 3. Normalización y detección de intención

RNF02: antes de procesar el mensaje se eliminan espacios innecesarios, se convierten caracteres a minúsculas y se normalizan tildes.

In [ ]:
def normalizar(texto):
    """Normaliza una entrada para facilitar su procesamiento."""
    if not isinstance(texto, str):
        return ""

    texto = texto.strip().lower()
    texto = unicodedata.normalize("NFD", texto)
    texto = "".join(c for c in texto if unicodedata.category(c) != "Mn")
    texto = re.sub(r"\s+", " ", texto)
    return texto


def detectar_intencion(texto):
    """Detecta la intención principal mediante palabras/frases clave."""
    texto = normalizar(texto)

    if texto in {"salir", "exit", "terminar", "cerrar", "adios", "chao"}:
        return "salir"

    if any(p in texto for p in ["disponibilidad", "habitaciones disponibles", "hay habitaciones", "reservar"]):
        return "disponibilidad"

    if any(p in texto for p in ["habitacion", "habitaciones", "servicios", "caracteristicas", "comodidades"]):
        return "habitaciones"

    if any(p in texto for p in ["comida", "desayuno", "restaurante", "alimentacion", "almuerzo", "cena"]):
        return "alimentacion"

    if "horario" in texto or "hora" in texto or "check in" in texto or "check-in" in texto:
        return "horarios"

    return "generica"


## 4. Respuestas del chatbot

Las fechas se solicitan para simular HU04. La disponibilidad utilizada es ficticia y no genera una reserva real.

In [ ]:
def respuesta_horarios():
    return (
        f"🕒 **Horarios del hotel**\n\n"
        f"• Check-in: **{HOTEL['check_in']}**\n"
        f"• Check-out: **{HOTEL['check_out']}**\n"
        f"• Desayuno: **{HOTEL['desayuno']}**\n"
        f"• Restaurante: **{HOTEL['restaurante']}**"
    )


def respuesta_alimentacion():
    return (
        "🍽️ **Opciones de alimentación**\n\n"
        f"• Desayuno: {HOTEL['desayuno']}\n"
        f"• Restaurante: {HOTEL['restaurante']}\n\n"
        "El desayuno puede incluirse en la reserva según la tarifa seleccionada. "
        "En este prototipo no se procesan pagos ni reservas reales."
    )


def respuesta_habitaciones():
    partes = ["🛏️ **Habitaciones y características**\n"]
    for datos in HABITACIONES.values():
        servicios = ", ".join(datos["servicios"])
        partes.append(
            f"**{datos['nombre']}**\n"
            f"• Capacidad: {datos['capacidad']} persona(s)\n"
            f"• Precio de referencia: ${datos['precio']:,} COP/noche\n"
            f"• Servicios: {servicios}\n"
        )
    return "\n".join(partes)


def respuesta_disponibilidad():
    disponibles = [
        datos["nombre"] for clave, datos in HABITACIONES.items()
        if DISPONIBILIDAD.get(clave, False)
    ]
    return (
        "📅 **Consulta de disponibilidad**\n\n"
        "Para el simulador, ingresa las fechas en formato **DD/MM/AAAA**. "
        "Por ejemplo: `20/10/2026 al 22/10/2026`.\n\n"
        "Habitaciones disponibles actualmente en el escenario de prueba:\n"
        + "\n".join(f"• {nombre}" for nombre in disponibles)
    )


def generar_respuesta(mensaje):
    """Genera una respuesta según la intención detectada."""
    intencion = detectar_intencion(mensaje)

    if intencion == "salir":
        return "👋 Gracias por utilizar el asistente del hotel. ¡Hasta pronto!", True
    if intencion == "horarios":
        return respuesta_horarios(), False
    if intencion == "alimentacion":
        return respuesta_alimentacion(), False
    if intencion == "habitaciones":
        return respuesta_habitaciones(), False
    if intencion == "disponibilidad":
        return respuesta_disponibilidad(), False

    return (
        "🤔 No encontré una respuesta específica para tu consulta.\n\n"
        "Puedes preguntarme por **horarios**, **desayuno**, **habitaciones** "
        "o **disponibilidad**. También puedes escribir **salir** para terminar."
    ), False


## 5. Lógica de conversación para Gradio

HU03 se implementa mediante el comando `salir` y un botón para limpiar/cerrar la interacción. La aplicación no ejecuta código arbitrario introducido por el usuario.

In [ ]:
def chat(mensaje, historial):
    """Procesa un mensaje y actualiza el historial de conversación."""
    if historial is None:
        historial = []

    mensaje = mensaje or ""
    if not mensaje.strip():
        return historial, ""

    respuesta, finalizar = generar_respuesta(mensaje)

    historial = historial + [
        {"role": "user", "content": mensaje.strip()},
        {"role": "assistant", "content": respuesta},
    ]

    return historial, ""


def limpiar_chat():
    return []


## 6. CSS y HTML de la interfaz web

La interfaz está pensada para ser embebida posteriormente en un sitio web. Gradio permite personalizar el aspecto mediante CSS y componentes HTML.

In [ ]:
CSS = r"""
body {
    background: #f4f7f6;
}

.hotel-header {
    text-align: center;
    padding: 18px;
    border-radius: 18px;
    background: linear-gradient(135deg, #173f35, #2e6b58);
    color: white;
    margin-bottom: 15px;
}

.hotel-header h1 {
    margin: 0;
    font-size: 30px;
}

.hotel-header p {
    margin: 6px 0 0;
    opacity: 0.9;
}

.info-box {
    border-radius: 14px;
    padding: 12px;
    background: white;
    border: 1px solid #dce5e1;
}

footer {display: none !important;}
"""

HEADER_HTML = """
<div class='hotel-header'>
    <h1>🏨 Hotel Montaña Verde</h1>
    <p>Asistente virtual para alojamiento y alimentación</p>
</div>
"""


## 7. Construcción de la aplicación Gradio

In [ ]:
with gr.Blocks(css=CSS, title="Asistente Hotel Montaña Verde") as demo:
    gr.HTML(HEADER_HTML)

    gr.Markdown(
        "### 💬 ¿En qué puedo ayudarte?\n"
        "Consulta horarios, alimentación, habitaciones o disponibilidad."
    )

    chatbot = gr.Chatbot(
        label="Conversac4ión",
        height=480,
        value=[
            {
                "role": "assistant",
                "content": "¡Hola! 👋 Soy el asistente virtual del hotel. ¿Quieres consultar horarios, alimentación, habitaciones o disponibilidad?"
            }
        ]
    )

    with gr.Row():
        mensaje = gr.Textbox(
            label="Mensaje",
            placeholder="Ejemplo: ¿Cuál es el horario de check-in?",
            scale=5,
        )
        enviar = gr.Button("Enviar", variant="primary", scale=1)

    with gr.Row():
        btn_horarios = gr.Button("🕒 Horarios")
        btn_comida = gr.Button("🍽️ Alimentación")
        btn_habitaciones = gr.Button("🛏️ Habitaciones")
        btn_disponibilidad = gr.Button("📅 Disponibilidad")
        limpiar = gr.Button("🗑️ Limpiar")

    gr.HTML(
        "<div class='info-box'><b>Privacidad:</b> prototipo académico. "
        "No ingreses datos personales, contraseñas o información bancaria.</div>"
    )

    enviar.click(chat, [mensaje, chatbot], [chatbot, mensaje])
    mensaje.submit(chat, [mensaje, chatbot], [chatbot, mensaje])
    limpiar.click(limpiar_chat, outputs=[chatbot])

    btn_horarios.click(lambda: chat("horario", []), outputs=[chatbot, mensaje])
    btn_comida.click(lambda: chat("desayuno", []), outputs=[chatbot, mensaje])
    btn_habitaciones.click(lambda: chat("habitaciones", []), outputs=[chatbot, mensaje])
    btn_disponibilidad.click(lambda: chat("disponibilidad", []), outputs=[chatbot, mensaje])

/tmp/ipykernel_1898/1892675513.py:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=CSS, title="Asistente Hotel Montaña Verde") as demo:


## 8. Ejecutar el chatbot

Al ejecutar esta celda, Colab mostrará la interfaz web de Gradio.

In [ ]:
demo.launch(share=True, debug=False)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5bb9d8761921c1e020.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 9. Pruebas de los requisitos funcionales

Esta celda permite verificar rápidamente las respuestas principales del simulador.

In [ ]:
casos_prueba = [
    ("¿Cuál es el horario de check-in?", "horarios"),
    ("Quiero saber si tienen desayuno", "alimentacion"),
    ("¿Qué habitaciones tienen?", "habitaciones"),
    ("¿Hay disponibilidad para mi reserva?", "disponibilidad"),
    ("salir", "salir"),
    ("xyz123", "generica"),
]

for mensaje_prueba, esperado in casos_prueba:
    resultado = detectar_intencion(mensaje_prueba)
    estado = "✅" if resultado == esperado else "❌"
    print(f"{estado} {mensaje_prueba!r} -> {resultado} (esperado: {esperado})")


## 10. Matriz de trazabilidad

| Requisito | Implementación |
|---|---|
| HU01 | Intención `horarios` y botón Horarios |
| HU02 | Intención `alimentacion` y botón Alimentación |
| HU03 | Comandos `salir` y `limpiar` |
| HU04 | Intención `disponibilidad` + solicitud de fechas |
| HU05 | Intención `habitaciones` + características |
| RF01 | `gr.Textbox` para entrada de texto |
| RF02 | Detección de `horario` / `hora` |
| RF03 | Detección de `comida` / `desayuno` / restaurante |
| RF04 | Respuesta genérica ante entradas no reconocidas |
| RF05 | Detección del comando `salir` |
| RNF01 | No se incluyen credenciales en el código |
| RNF02 | Función `normalizar()` |
| RNF03 | Prototipo sin captura de datos sensibles; debe complementarse con controles legales/técnicos en producción |
| RNF04 | Código dividido por responsabilidades y documentado |
| RNF05 | Respuestas estructuradas y lenguaje sencillo |


## 11. Próximos pasos para convertir el simulador en un sistema real

1. Conectar `respuesta_disponibilidad()` con una base de datos de habitaciones y reservas.
2. Validar fechas reales y evitar reservas superpuestas.
3. Separar backend, frontend y configuración.
4. Si se incorpora una API de IA, guardar su clave en **Colab Secrets** o variables de entorno, nunca en el notebook.
5. Incorporar autenticación y autorización si se agregan funciones administrativas.
6. Aplicar controles de protección de datos personales acordes con la normativa colombiana aplicable (por ejemplo, Ley 1581 de 2012 y sus reglamentaciones) antes de manejar datos reales.
7. Para publicar el chatbot en el sitio web, utilizar el enlace/servicio de Gradio o desplegar la aplicación en un servidor. `share=True` es adecuado para demostraciones, no como arquitectura definitiva de producción.
8. Registrar únicamente la información necesaria y definir políticas de retención, acceso y eliminación de datos.